In [8]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [9]:
##Load trained model , scaler picke , onehot
model=load_model('model.h5')

#Load encoder and decoder
with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_geo = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [10]:
# Sample customer input
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000.0,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000.0
}

print(input_data)

{'CreditScore': 600, 'Geography': 'France', 'Gender': 'Male', 'Age': 40, 'Tenure': 3, 'Balance': 60000.0, 'NumOfProducts': 2, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 50000.0}


In [12]:
import pickle

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

print("OneHot Encoder loaded successfully!")

OneHot Encoder loaded successfully!


In [13]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000.0,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000.0
}

print(input_data['Geography'])

France


In [14]:
geo_encoded = onehot_encoder_geo.transform(
    [[input_data['Geography']]]
).toarray()

print("One-Hot Encoded Geography:")
print(geo_encoded)

One-Hot Encoded Geography:
[[1. 0. 0.]]


c:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [15]:
import pandas as pd

geo_encoded = onehot_encoder_geo.transform(
    pd.DataFrame({'Geography': [input_data['Geography']]})
).toarray()

print("One-Hot Encoded Geography:")
print(geo_encoded)

One-Hot Encoded Geography:
[[1. 0. 0.]]


In [16]:
import pandas as pd

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

print(geo_encoded_df)

   Geography_France  Geography_Germany  Geography_Spain
0               1.0                0.0              0.0


In [19]:
# Encode Gender
gender_encoded = label_encoder_gender.transform(
    [input_data['Gender']]
)

print("Encoded Gender:", gender_encoded[0])

NameError: name 'label_encoder_gender' is not defined

In [20]:
import pickle

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

print("Gender encoder loaded successfully!")

Gender encoder loaded successfully!


In [22]:
print(label_encoder_gender)
print(hasattr(label_encoder_gender, 'classes_'))

LabelEncoder()
False


In [23]:
from sklearn.preprocessing import LabelEncoder

label_encoder_gender = LabelEncoder()
label_encoder_gender.fit(['Female', 'Male'])

print("Classes:", label_encoder_gender.classes_)

Classes: ['Female' 'Male']


In [24]:
gender_encoded = label_encoder_gender.transform(
    [input_data['Gender']]
)

print("Encoded Gender:", gender_encoded[0])

Encoded Gender: 1


In [25]:
import pandas as pd

# Numerical features
numerical_data = pd.DataFrame([{
    'CreditScore': input_data['CreditScore'],
    'Age': input_data['Age'],
    'Tenure': input_data['Tenure'],
    'Balance': input_data['Balance'],
    'NumOfProducts': input_data['NumOfProducts'],
    'HasCrCard': input_data['HasCrCard'],
    'IsActiveMember': input_data['IsActiveMember'],
    'EstimatedSalary': input_data['EstimatedSalary']
}])

# Combine numerical and encoded features
input_data_combined = pd.concat([
    numerical_data,
    pd.DataFrame({'Gender': gender_encoded}),
    pd.DataFrame(
        geo_encoded,
        columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
    )
], axis=1)

print(input_data_combined)

   CreditScore  Age  Tenure  Balance  NumOfProducts  HasCrCard  \
0          600   40       3  60000.0              2          1   

   IsActiveMember  EstimatedSalary  Gender  Geography_France  \
0               1          50000.0       1               1.0   

   Geography_Germany  Geography_Spain  
0                0.0              0.0  


In [26]:
expected_columns = [
    'CreditScore', 'Gender', 'Age', 'Tenure', 'Balance',
    'NumOfProducts', 'HasCrCard', 'IsActiveMember',
    'EstimatedSalary', 'Geography_France',
    'Geography_Germany', 'Geography_Spain'
]

input_data_combined = input_data_combined[expected_columns]

print("Final input shape:", input_data_combined.shape)

Final input shape: (1, 12)


In [27]:
input_scaled = scaler.transform(input_data_combined)

print("Scaled input:")
print(input_scaled)

Scaled input:
[[-0.52806744  1.00803226  0.06719428 -0.6396364  -0.07404552  0.78576395
   0.66030894  0.97897106 -0.87434415  1.00828431 -0.57138416 -0.58966846]]


In [28]:
prediction = model.predict(input_scaled)

print("Churn probability:", prediction[0][0])

if prediction[0][0] >= 0.5:
    print("Customer is likely to leave the bank.")
else:
    print("Customer is likely to stay with the bank.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 264ms/step
Churn probability: 0.123783916
Customer is likely to stay with the bank.


In [29]:
import pickle
from sklearn.preprocessing import LabelEncoder

label_encoder_gender = LabelEncoder()
label_encoder_gender.fit(["Female", "Male"])

with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(label_encoder_gender, file)

print("Gender encoder saved successfully!")
print(label_encoder_gender.classes_)

Gender encoder saved successfully!
['Female' 'Male']
